In [1]:
import pandas as pd
from dataclasses import dataclass
from typing import Dict, List, Any, Optional

class IFCTDatabase:
    """
    Read-only IFCT interface.

    Expects a CSV with at least these columns (per 100g):
      - ingredient
      - sodium_mg_per_100g
      - potassium_mg_per_100g
      - protein_g_per_100g
      - carbs_g_per_100g

    You can add more nutrients later (phosphorus, calories, etc.).
    """
    def __init__(self, ifct_csv_path: str):
        self.df = pd.read_csv(ifct_csv_path)

        required = {
            "ingredient",
            "sodium_mg_per_100g",
            "potassium_mg_per_100g",
            "protein_g_per_100g",
            "carbs_g_per_100g",
        }
        missing = required - set(self.df.columns)
        if missing:
            raise ValueError(f"IFCT CSV missing columns: {sorted(missing)}")

        self.df["ingredient_norm"] = self.df["ingredient"].astype(str).str.lower().str.strip()
        self.idx = self.df.set_index("ingredient_norm")

    def get_nutrients_per_100g(self, ingredient: str) -> Dict[str, float]:
        key = ingredient.lower().strip()
        if key not in self.idx.index:
            raise KeyError(f"Ingredient not found in IFCT: {ingredient}")

        row = self.idx.loc[key]
        return {
            "sodium_mg": float(row["sodium_mg_per_100g"]),
            "potassium_mg": float(row["potassium_mg_per_100g"]),
            "protein_g": float(row["protein_g_per_100g"]),
            "carbs_g": float(row["carbs_g_per_100g"]),
        }


In [2]:
@dataclass
class DailyBudget:
    """
    Remaining budget for the day (not total recommended intake).
    These should come from:
    - guideline targets
    - minus what the user already consumed today (if tracking)
    """
    sodium_mg_remaining: float
    potassium_mg_remaining: float
    protein_g_remaining: float
    carbs_g_remaining: float

@dataclass
class PortionDecision:
    ingredient: str
    max_grams: float
    label: str                 # "Allowed" | "Half portion" | "Avoid"
    explanation: str
    nutrient_load_at_max: Dict[str, float]
    binding_constraint: str     # which nutrient constrained the portion most


In [3]:
class PortionRecommender:
    """
    Deterministic portion recommendation engine.
    Inputs:
      - risk_levels from Model #1:
          {
            "sodium_sensitivity": "low/moderate/high",
            "potassium_sensitivity": "low/moderate/high",
            "protein_restriction": "low/moderate/high"
          }
      - IFCT nutrient densities
      - remaining daily budgets

    Output:
      PortionDecision with max grams + label + explanation.

    Key safety properties:
      - No hallucinated nutrition facts (IFCT only)
      - No black-box learning (pure constrained logic)
    """

    # Risk -> fraction of remaining daily budget allowed for a SINGLE ingredient
    # (conservative by design, tune later with clinician feedback)
    RISK_FRACTION = {
        "low": 0.40,
        "moderate": 0.25,
        "high": 0.10,
    }

    def __init__(
        self,
        ifct: IFCTDatabase,
        default_cap_g: float = 300.0,
        half_portion_threshold_g: float = 75.0,
        avoid_threshold_g: float = 1.0
    ):
        self.ifct = ifct
        self.default_cap_g = float(default_cap_g)
        self.half_portion_threshold_g = float(half_portion_threshold_g)
        self.avoid_threshold_g = float(avoid_threshold_g)

    @staticmethod
    def _grams_from_budget(nutrient_per_100g: float, remaining_budget: float, frac: float) -> float:
        """
        Compute maximum grams such that nutrient load <= (remaining_budget * frac)
        grams = (allowed / nutrient_per_100g) * 100
        """
        if nutrient_per_100g <= 0:
            return float("inf")  # not constrained by this nutrient
        allowed = max(0.0, remaining_budget) * frac
        grams = (allowed / nutrient_per_100g) * 100.0
        return max(0.0, grams)

    def recommend_one(
        self,
        ingredient: str,
        risk_levels: Dict[str, str],
        budget: DailyBudget,
        carb_risk: str = "moderate"  # you can connect DM-specific risk later if you want
    ) -> PortionDecision:
        n = self.ifct.get_nutrients_per_100g(ingredient)

        sod_risk = risk_levels.get("sodium_sensitivity", "moderate")
        pot_risk = risk_levels.get("potassium_sensitivity", "moderate")
        pro_risk = risk_levels.get("protein_restriction", "moderate")

        # Safety: normalize unexpected strings
        def norm_risk(r: str) -> str:
            r = (r or "moderate").lower().strip()
            return r if r in self.RISK_FRACTION else "moderate"

        sod_risk = norm_risk(sod_risk)
        pot_risk = norm_risk(pot_risk)
        pro_risk = norm_risk(pro_risk)
        carb_risk = norm_risk(carb_risk)

        # Compute grams allowed by each constraint
        g_sod = self._grams_from_budget(n["sodium_mg"], budget.sodium_mg_remaining, self.RISK_FRACTION[sod_risk])
        g_pot = self._grams_from_budget(n["potassium_mg"], budget.potassium_mg_remaining, self.RISK_FRACTION[pot_risk])
        g_pro = self._grams_from_budget(n["protein_g"], budget.protein_g_remaining, self.RISK_FRACTION[pro_risk])
        g_carb = self._grams_from_budget(n["carbs_g"], budget.carbs_g_remaining, self.RISK_FRACTION[carb_risk])

        constraints = {
            "sodium": g_sod,
            "potassium": g_pot,
            "protein": g_pro,
            "carbs": g_carb,
        }
        binding = min(constraints, key=constraints.get)

        max_g = min(constraints.values())
        max_g = min(max_g, self.default_cap_g)  # practical cap

        # Label policy
        if max_g <= self.avoid_threshold_g:
            label = "Avoid"
        elif max_g <= self.half_portion_threshold_g:
            label = "Half portion"
        else:
            label = "Allowed"

        # Nutrient load at max grams
        factor = max_g / 100.0
        load = {
            "sodium_mg": n["sodium_mg"] * factor,
            "potassium_mg": n["potassium_mg"] * factor,
            "protein_g": n["protein_g"] * factor,
            "carbs_g": n["carbs_g"] * factor,
        }

        # Clinical explanation (binding constraint driven)
        reasons = {
            "sodium": f"limited due to sodium sensitivity ({sod_risk})",
            "potassium": f"limited due to potassium sensitivity ({pot_risk})",
            "protein": f"limited due to protein restriction ({pro_risk})",
            "carbs": f"limited due to carbohydrate load ({carb_risk})",
        }
        explanation = (
            f"{label}: max ~{max_g:.0f}g because this ingredient is primarily "
            f"{reasons[binding]}. Nutrient values are computed from IFCT per-100g composition."
        )

        return PortionDecision(
            ingredient=ingredient,
            max_grams=float(max_g),
            label=label,
            explanation=explanation,
            nutrient_load_at_max=load,
            binding_constraint=binding,
        )

    def recommend_many(
        self,
        ingredients: List[str],
        risk_levels: Dict[str, str],
        budget: DailyBudget,
        carb_risk: str = "moderate"
    ) -> List[PortionDecision]:
        out = []
        for ing in ingredients:
            try:
                out.append(self.recommend_one(ing, risk_levels, budget, carb_risk=carb_risk))
            except KeyError:
                out.append(
                    PortionDecision(
                        ingredient=ing,
                        max_grams=0.0,
                        label="Avoid",
                        explanation="Avoid: ingredient not found in IFCT (cannot verify nutrient composition safely).",
                        nutrient_load_at_max={"sodium_mg": 0.0, "potassium_mg": 0.0, "protein_g": 0.0, "carbs_g": 0.0},
                        binding_constraint="unknown",
                    )
                )
        return out


In [ ]:
# ---- Demo IFCT file (create only if you don't have one yet) ----
import os

ifct_path = "ifct_sample.csv"
if not os.path.exists(ifct_path):
    demo = pd.DataFrame([
        {"ingredient":"tomato",  "sodium_mg_per_100g":5,  "potassium_mg_per_100g":237, "protein_g_per_100g":0.9, "carbs_g_per_100g":3.9},
        {"ingredient":"spinach", "sodium_mg_per_100g":79, "potassium_mg_per_100g":558, "protein_g_per_100g":2.9, "carbs_g_per_100g":3.6},
        {"ingredient":"banana",  "sodium_mg_per_100g":1,  "potassium_mg_per_100g":358, "protein_g_per_100g":1.1, "carbs_g_per_100g":22.8},
        {"ingredient":"curd",    "sodium_mg_per_100g":50, "potassium_mg_per_100g":155, "protein_g_per_100g":3.5, "carbs_g_per_100g":4.7},
    ])
    demo.to_csv(ifct_path, index=False)

ifct = IFCTDatabase(ifct_path)
portion_model = PortionRecommender(ifct)

# Example risk_levels 
risk_levels = {
    "sodium_sensitivity": "high",
    "potassium_sensitivity": "high",
    "protein_restriction": "moderate",
}

# Example remaining daily nutrient budgets
budget = DailyBudget(
    sodium_mg_remaining=600,    
    potassium_mg_remaining=900,  
    protein_g_remaining=35,
    carbs_g_remaining=120
)

ingredients = ["tomato", "spinach", "banana", "curd", "unknown_item"]

decisions = portion_model.recommend_many(ingredients, risk_levels, budget, carb_risk="moderate")

pd.DataFrame([{
    "ingredient": d.ingredient,
    "label": d.label,
    "max_grams": round(d.max_grams, 1),
    "binding": d.binding_constraint,
    "explanation": d.explanation
} for d in decisions])


,ingredient,label,max_grams,binding,explanation
0,tomato,Half portion,38.0,potassium,Half portion: max ~38g because this ingredient...
1,spinach,Half portion,16.1,potassium,Half portion: max ~16g because this ingredient...
2,banana,Half portion,25.1,potassium,Half portion: max ~25g because this ingredient...
3,curd,Half portion,58.1,potassium,Half portion: max ~58g because this ingredient...
4,unknown_item,Avoid,0.0,unknown,Avoid: ingredient not found in IFCT (cannot ve...
